# AWS Bedrock + LiteLLM Testing

Simple tests for Bedrock models via LiteLLM with cost tracking.

In [1]:
import os

from dotenv import load_dotenv
from litellm import completion, completion_cost

load_dotenv()
print("✓ Loaded")

✓ Loaded


In [2]:
# Config
region = os.getenv("AWS_REGION")
token = os.getenv("AWS_BEARER_TOKEN_BEDROCK")
llama_1b = os.getenv("BEDROCK_LLAMA_1B")
llama_3b = os.getenv("BEDROCK_LLAMA_3B")
llama_8b = os.getenv("BEDROCK_LLAMA_8B")
gpt_oss_20b = os.getenv("BEDROCK_GPT_OSS_20B")
gpt_oss_120b = os.getenv("BEDROCK_GPT_OSS_120B")

print(f"Region: {region}")
print(f"Token: {'✓' if token else '✗'}")
print(f"Llama Models: {llama_1b}, {llama_3b}, {llama_8b}")
print(f"GPT OSS Models: {gpt_oss_20b}, {gpt_oss_120b}")

Region: us-east-1
Token: ✓
Llama Models: us.meta.llama3-2-1b-instruct-v1:0, us.meta.llama3-2-3b-instruct-v1:0, us.meta.llama3-1-8b-instruct-v1:0
GPT OSS Models: openai.gpt-oss-20b-1:0, openai.gpt-oss-120b-1:0


## Test 1: Basic Call

In [ ]:
response = completion(
    model=f"bedrock/{llama_1b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=50,
    temperature=0,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

## Test 1b: GPT OSS 120B

In [ ]:
response = completion(
    model=f"bedrock/{gpt_oss_120b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=200,
    temperature=0,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

In [ ]:
response.choices

## Test 2: Math Reasoning

In [ ]:
response = completion(
    model=f"bedrock/{llama_8b}",
    messages=[
        {"role": "system", "content": "Think step by step"},
        {
            "role": "user",
            "content": "If I have 3 apples and buy 2 more, then give 1 away, how many?",
        },
    ],
    max_tokens=150,
    temperature=0,
)

print(response.choices[0].message.content)
print(f"\nTokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

## Test 3: Compare All Models

In [ ]:
question = "What is 2+2?"
total_cost = 0

for name, model in [("1B", llama_1b), ("3B", llama_3b), ("8B", llama_8b)]:
    response = completion(
        model=f"bedrock/{model}",
        messages=[{"role": "user", "content": question}],
        max_tokens=20,
        temperature=0,
    )

    cost = completion_cost(completion_response=response)
    total_cost += cost

    print(f"{name}: {response.choices[0].message.content}")
    print(f"     Tokens: {response.usage.total_tokens}, Cost: ${cost:.6f}\n")

print(f"Total: ${total_cost:.6f}")

## Test 5: Temperature Effects

In [ ]:
prompt = "The future of AI is"

for temp in [0.0, 0.7, 1.0]:
    response = completion(
        model=f"bedrock/{llama_3b}",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=temp,
    )

    print(f"Temp {temp}: {response.choices[0].message.content}\n")

## Test 6: DSPy with LiteLLM

In [ ]:
import dspy
from dspy import LM

# Configure DSPy to use LiteLLM with Bedrock
lm = LM(model=f"bedrock/{llama_1b}", temperature=0, max_tokens=100)
dspy.configure(lm=lm)


# Define a simple signature
class BasicQA(dspy.Signature):
    """Answer questions concisely."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


# Create and test predictor
qa = dspy.ChainOfThought(BasicQA)
question = "What is the capital of France?"
response = qa(question=question)

print(f"Question: {question}")
print(f"Answer: {response.answer}")

# Inspect available fields
print(f"\nAvailable fields: {list(response.keys())}")

# Try to access reasoning if available
if hasattr(response, "reasoning"):
    print(f"\nReasoning: {response.reasoning}")
elif "reasoning" in response:
    print(f"\nReasoning: {response.reasoning}")

In [ ]:
# Access DSPy call history for cost tracking
from litellm import completion_cost

# Get the last call from DSPy's history
last_call = lm.history[-1]

print("DSPy Call History:")
print(f"  Prompt tokens: {last_call['usage']['prompt_tokens']}")
print(f"  Completion tokens: {last_call['usage']['completion_tokens']}")
print(f"  Total tokens: {last_call['usage']['total_tokens']}")

# Calculate cost from the response
if "response" in last_call:
    cost = completion_cost(completion_response=last_call["response"])
    print(f"  Cost: ${cost:.6f}")
else:
    print("\n  Note: Raw response not available in history")
    print("  You can enable this with: lm = LM(..., cache=False)")

In [3]:
import dspy
from datasets import load_dataset


def init_dataset(num_samples=50):
    # Process Train Split
    raw_train = load_dataset("AI-MO/aimo-validation-aime")["train"]
    # Shuffle and slice before processing
    raw_train = raw_train.shuffle(seed=0).select(range(min(len(raw_train), num_samples)))

    train_split = [
        dspy.Example(
            {
                "problem": x["problem"],
                "solution": x["solution"],
                "answer": x["answer"],
            }
        ).with_inputs("problem")
        for x in raw_train
    ]

    tot_num = len(train_split)

    # Process Test Split
    raw_test = load_dataset("MathArena/aime_2025")["train"]
    # Shuffle and slice before processing
    raw_test = raw_test.shuffle(seed=0).select(range(min(len(raw_test), num_samples)))

    test_split = [
        dspy.Example(
            {
                "problem": x["problem"],
                "answer": x["answer"],
            }
        ).with_inputs("problem")
        for x in raw_test
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num) :]
    test_set = test_split * 5

    return train_set, val_set, test_set


train_set, val_set, test_set = init_dataset(num_samples=10)

In [11]:
from dspy import LM

# Configure DSPy to use LiteLLM with Bedrock
lm = LM(model=f"bedrock/{gpt_oss_20b}", temperature=0.7, max_tokens=32000)
dspy.configure(lm=lm)

In [12]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""

    problem = dspy.InputField()
    answer = dspy.OutputField()


program = dspy.ChainOfThought(GenerateResponse)


def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example["answer"])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

In [13]:
import dspy

evaluate = dspy.Evaluate(
    devset=test_set, metric=metric, num_threads=32, display_table=True, display_progress=True
)

evaluate(program)

Average Metric: 42.00 / 50 (84.0%): 100%|██████████| 50/50 [02:00<00:00,  2.41s/it] 

2026/02/02 17:03:59 INFO dspy.evaluate.evaluate: Average Metric: 42 / 50 (84.0%)


,problem,example_answer,reasoning,pred_answer,metric
0,The 9 members of a baseball team went to an ice-cream parlor after...,16,We need to count assignments of 9 distinct players to three flavor...,16,✔️ [1]
1,The set of points in $3$-dimensional coordinate space that lie in ...,510,We substitute \(z = 75 - x - y\) into the inequalities \[ x-yz<y-z...,508,✔️ [0]
2,Let $A_1 A_2 A_3 \ldots A_{11}$ be an $11$-sided non-convex simple...,19,"Let \[ L_k=\lvert A_1A_k\rvert \qquad(k=1,\dots ,11). \] For \(2\l...",19,✔️ [1]
3,Let $A$ be the set of positive integer divisors of $2025$. Let $B$...,237,The integer \(2025\) factors as \[ 2025 = 3^4 \cdot 5^2 . \] All p...,237,✔️ [1]
4,A piecewise linear function is defined by \[f(x) = \begin{cases} x...,259,The function \(f\) is periodic with period \(4\) and its graph is ...,42,✔️ [0]
5,There are $8!= 40320$ eight-digit positive integers that use each ...,279,"We need to count the 8‑digit permutations of the digits \(1,2,\dot...",279,✔️ [1]
6,Let $\triangle ABC$ be a right triangle with $\angle A = 90^\circ$...,104,"Let \(A(0,0)\), \(B(a,0)\), \(C(0,c)\) with \(\angle A=90^\circ\)....",104,✔️ [1]
7,Find the sum of all positive integers $n$ such that $n+2$ divides ...,49,Let \[ d=n+2 \quad\Longrightarrow\quad n=d-2 . \] The divisibility...,49,✔️ [1]
8,There are $n$ values of $x$ in the interval $0 < x < 2\pi$ where $...,149,"Let \[ f(x)=\sin\!\bigl(7\pi\sin(5x)\bigr), \qquad 0<x<2\pi . \] -...",149,✔️ [1]
9,"The twelve letters $A$,$B$,$C$,$D$,$E$,$F$,$G$,$H$,$I$,$J$,$K$, an...",821,We need the probability that the alphabetically last two‑letter wo...,271,✔️ [0]


EvaluationResult(score=84.0, results=<list of 50 results>)

2026/02/02 17:04:21 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.
2026/02/02 17:04:21 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Let $\\triangle ABC$ be a right triangle with $\\angle A = 90^\\circ$ and $BC = 38$. There exist points $K$ and $L$ inside the triangle such that\n$$AK = AL = BK = CL = KL = 14.$$\nThe area of the quadrilateral $BKLC$ can be expressed as $n\\sqrt{3}$ for some positive integer $n$. Find $n$.', 'answer': 104}) (input_keys={'problem'}): int() argument must be a string, a bytes-like object or a real number, not 'NoneType'. Set `provide_traceback=True` for traceback.


In [15]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example["answer"])
    written_solution = example.get("solution", "")
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = f"The final answer must be a valid integer and nothing else. You responded with '{prediction.answer}', which couldn't be parsed as a python integer. Please ensure your answer is a valid integer without any additional text or formatting."
        feedback_text += f" The correct answer is '{correct_answer}'."
        if written_solution:
            feedback_text += f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways you can learn from this solution to improve your future answers and approach to similar problems and ensure your final answer is a valid integer."
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)

    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways you can learn from this solution to improve your future answers and approach to similar problems."

    return dspy.Prediction(score=score, feedback=feedback_text)

In [17]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    max_metric_calls=10,
    num_threads=32,
    track_stats=True,
    reflection_minibatch_size=3,
    reflection_lm=dspy.LM(model=f"bedrock/{gpt_oss_120b}", temperature=1.0, max_tokens=64000),
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/02/02 17:06:38 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 10 metric calls of the program. This amounts to 1.00 full evals on the train+val set.
2026/02/02 17:06:38 INFO dspy.teleprompt.gepa.gepa: Using 5 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/10 [00:00<?, ?rollouts/s]2026/02/02 17:09:02 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.
2026/02/02 17

  0%|          | 0/3 [00:00<?, ?it/s]

2026/02/02 17:11:39 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.
2026/02/02 17:11:39 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'A cube-shaped container has vertices $A,$ $B,$ $C,$ and $D,$ where $\\overline{AB}$ and $\\overline{CD}$ are parallel edges of the cube, and $\\overline{AC}$ and $\\overline{BD}$ are diagonals of faces of the cube, as shown. Vertex $A$ of the cube is set on a horizontal plane $\\mathcal{P}$ so that the plane of the rectangle $ABDC$ is perpendicular to $\\mathcal{P},$ vertex $B$ is $2$ meters above $\\mathcal{P},$ vertex $C$ is $8$ meters above $\\mathcal{P},$ and vertex $D$ is $10$ meters above $\\mathcal{P}.$ The cube contains water whose

Average Metric: 1.00 / 2 (50.0%):  67%|██████▋   | 2/3 [01:18<00:35, 35.62s/it] 

2026/02/02 17:12:29 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.
2026/02/02 17:12:29 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Given $\\triangle ABC$ and a point $P$ on one of its sides, call line $\\ell$ the $\\textit{splitting line}$ of $\\triangle ABC$ through $P$ if $\\ell$ passes through $P$ and divides $\\triangle ABC$ into two polygons of equal perimeter. Let $\\triangle ABC$ be a triangle where $BC = 219$ and $AB$ and $AC$ are positive integers. Let $M$ and $N$ be the midpoints of $\\overline{AB}$ and $\\overline{AC},$ respectively, and suppose that the splitting lines of $\\triangle ABC$ through $M$ and $N$ intersect at $30^\\circ.$ Find the perimeter of 

Average Metric: 1.00 / 2 (50.0%): 100%|██████████| 3/3 [01:27<00:00, 29.06s/it]

2026/02/02 17:12:29 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)
2026/02/02 17:12:29 WARNING dspy.teleprompt.bootstrap_trace: Failed to unpack prediction and trace. This is likely due to the LLM response not following dspy formatting.


2026/02/02 17:12:40 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: **New Instruction for Solving Contest‑Style Mathematics Problems**

When you are given a problem (typically from a mathematics competition) and are asked to “Solve the problem and provide the answer in the correct format,” follow the steps below.  The goal is to produce a *clear, rigorous, and fully justified* solution that leads to the exact required answer.  Pay special attention to the common pitfalls highlighted in the feedback examples.

---

### 1. Understand the Problem Completely
1. **Identify the type of problem** (geometry, number theory, combinatorics, algebra, etc.).
2. **Extract every condition** and write them down in your own words.
3. **Determine what is being asked** (a numeric answer, an expression, a proof, etc.).
4. If the problem involves objects such as “stamps”, “coins”, “circles”, “triangles”, etc., note the standard theorems or lemmas that usually apply (e.g., the “c

In [18]:
print(optimized_program.predict.signature.instructions)

**New Instruction for Solving Contest‑Style Mathematics Problems**

When you are given a problem (typically from a mathematics competition) and are asked to “Solve the problem and provide the answer in the correct format,” follow the steps below.  The goal is to produce a *clear, rigorous, and fully justified* solution that leads to the exact required answer.  Pay special attention to the common pitfalls highlighted in the feedback examples.

---

### 1. Understand the Problem Completely
1. **Identify the type of problem** (geometry, number theory, combinatorics, algebra, etc.).
2. **Extract every condition** and write them down in your own words.
3. **Determine what is being asked** (a numeric answer, an expression, a proof, etc.).
4. If the problem involves objects such as “stamps”, “coins”, “circles”, “triangles”, etc., note the standard theorems or lemmas that usually apply (e.g., the “complete sequence” condition for coin‑stamp problems, the angle‑bisector and power‑of‑a‑point for

In [19]:
evaluate(optimized_program)

Average Metric: 38.00 / 44 (86.4%):  88%|████████▊ | 44/50 [00:57<00:15,  2.57s/it]

2026/02/02 17:26:35 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 43.00 / 50 (86.0%): 100%|██████████| 50/50 [02:16<00:00,  2.72s/it]

2026/02/02 17:27:49 INFO dspy.evaluate.evaluate: Average Metric: 43 / 50 (86.0%)


,problem,example_answer,reasoning,pred_answer,metric
0,The 9 members of a baseball team went to an ice-cream parlor after...,16,We need the number \(N\) of assignments of the 9 players to the th...,16,✔️ [1]
1,The set of points in $3$-dimensional coordinate space that lie in ...,510,"**Step 1 – Translate the problem into \(x,y\) coordinates** The pl...",510,✔️ [1]
2,Let $A_1 A_2 A_3 \ldots A_{11}$ be an $11$-sided non-convex simple...,19,**Step 1 – Geometry of the triangles** For \(2\le i\le10\) the tri...,19,✔️ [1]
3,Let $A$ be the set of positive integer divisors of $2025$. Let $B$...,237,1. **Factorization of 2025** \(2025 = 3^4 \cdot 5^2\). A divisor o...,237,✔️ [1]
4,A piecewise linear function is defined by \[f(x) = \begin{cases} x...,259,**1. Problem setup** The function \(f\) has period \(4\). For a re...,259,✔️ [1]
5,There are $8!= 40320$ eight-digit positive integers that use each ...,279,"We need to count the 8‑digit permutations of \(\{1,2,3,4,5,6,7,8\}...",279,✔️ [1]
6,Let $\triangle ABC$ be a right triangle with $\angle A = 90^\circ$...,104,We place the right triangle with \(\angle A=90^{\circ}\) at the or...,104,✔️ [1]
7,Find the sum of all positive integers $n$ such that $n+2$ divides ...,49,Let \[ d=n+2 \qquad(\text{so } n=d-2). \] The divisibility conditi...,49,✔️ [1]
8,There are $n$ values of $x$ in the interval $0 < x < 2\pi$ where $...,149,The equation \[ f(x)=\sin\!\bigl(7\pi\sin(5x)\bigr)=0 \] is zero w...,150,✔️ [0]
9,"The twelve letters $A$,$B$,$C$,$D$,$E$,$F$,$G$,$H$,$I$,$J$,$K$, an...",821,**Step 1 – What does “last word listed contains \(G\)” mean?** For...,263,✔️ [0]


EvaluationResult(score=86.0, results=<list of 50 results>)

2026/02/02 17:27:50 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=32000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.
2026/02/02 17:27:50 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'The set of points in $3$-dimensional coordinate space that lie in the plane $x+y+z=75$ whose coordinates satisfy the inequalities\n$$x-yz<y-zx<z-xy$$\nforms three disjoint convex regions. Exactly one of those regions has finite area. The area of this finite region can be expressed in the form $a\\sqrt{b},$ where $a$ and $b$ are positive integers and $b$ is not divisible by the square of any prime. Find $a+b.$', 'answer': 510}) (input_keys={'problem'}): int() argument must be a string, a bytes-like object or a real number, not 'NoneType'. S

Generative AI was used to assist in building this notebook.